### POS-Tagging and Parsing

#### 1. Part-of-Speech (POS) Tagging
Part-of-Speech (POS) tagging is the process of labeling each word in a text with its corresponding grammatical role (e.g., noun, verb, adjective, adverb) based on its syntactic context and lexical definition.

*   **Standard vs. Social Media (Twitter) Data:**
    *   **Standard Text:** Follows formal grammar, standard orthography, and structured punctuation (e.g., Wall Street Journal, Penn Treebank).
    *   **Twitter / Social Media Text:** Characterized by informal language, misspellings, emojis, hashtags (`#NLP`), user mentions (`@username`), abbreviations (`rn`, `smh`), and URLs. Specialized tokenizers (such as NLTK's `TweetTokenizer`) are required to prevent entity splitting before tag assignment.
*   **Tagsets:**
    *   **Penn Treebank (Fine-Grained):** Contains detailed tags differentiating inflections (e.g., `NN` for singular noun, `NNS` for plural noun, `VBZ` for 3rd person singular present verb, `JJ` for adjective).
    *   **Universal POS (Coarse-Grained):** Standardized, simplified tagset across languages containing broad categories such as `NOUN`, `VERB`, `ADJ`, `ADV`, `PRON`, and `PUNCT`.

---

#### 2. Chunking (Shallow Parsing)
Chunking segments a sentence into non-overlapping, syntactically related phrases rather than building a fully nested parse tree.

*   **Grammar Specifications:** Defined using regular expressions over POS tags.
    *   **Noun Phrase (NP Chunk):** Typically includes an optional determiner, zero or more adjectives, and one or more nouns:
        $$\text{NP: } \{<\text{DT}>?<\text{JJ}>*<\text{NN.*}>+\}$$
    *   **Verb Phrase (VP Chunk):** Typically includes an optional adverb followed by a main or auxiliary verb:
        $$\text{VP: } \{<\text{RB.*}>*<\text{VB.*}>+\}$$
*   **Chinking:** The process of defining what *not* to include in a chunk. The parser selects an entire block of tokens and carves out ("chinks") specified patterns (e.g., removing prepositions or conjunctions).
*   **IOB Tagging Scheme:**
    *   **B (Begin):** Indicates that the token marks the start of a new chunk.
    *   **I (Inside):** Indicates that the token is inside an active chunk.
    *   **O (Outside):** Indicates that the token does not belong to any chunk.

---

#### 3. Syntactic Parsing (Full Parsing)
Parsing analyzes a sequence of tokens to uncover its underlying syntactic and grammatical hierarchy.

*   **Constituency Parsing (Phrase Structure Parsing):** Breaks a sentence into nested structural sub-phrases governed by Context-Free Grammars (CFG). Identifies constituents like Noun Phrases ($NP$) and Verb Phrases ($VP$).
*   **Dependency Parsing:** Models syntactic structure through directed binary grammatical relations directly between words (a head and its dependents), such as nominal subject (`nsubj`), direct object (`dobj`), and adjectival modifier (`amod`).

# Environment Setup

In [1]:
# Install dependencies
!pip install nltk spacy pandas
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 85.8 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
# Import modules & download NLTK data
import nltk
from nltk.tokenize import TweetTokenizer, word_tokenize
from nltk.corpus import stopwords
import pandas as pd
import spacy
from spacy import displacy

# Download required NLTK taggers and chunk models
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('universal_tagset')

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package universal_tagset to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!


True

# Categorizing and Tagging Words in Twitter Data

In [3]:
# Tokenizing and Tagging Twitter Data
sample_tweets = [
    "Loving the new #PyTorch 2.0 release! @OpenAI models run so fast rn 🔥🚀",
    "Can't believe the flight got delayed again... smh @Delta #traveldiaries",
    "NLP is amazing! Check out https://huggingface.co for open-source models."
]

# Initialize TweetTokenizer: preserves handles, URLs, and reduces repeated characters
tweet_tokenizer = TweetTokenizer(preserve_case=False, strip_handles=False, reduce_len=True)

tweet_tag_records = []

for idx, tweet in enumerate(sample_tweets, start=1):
    # 1. Tokenize using Twitter-aware tokenizer
    tokens = tweet_tokenizer.tokenize(tweet)
    
    # 2. Tag with Penn Treebank tagset & Universal tagset
    tags_ptb = nltk.pos_tag(tokens)
    tags_univ = nltk.pos_tag(tokens, tagset='universal')
    
    for (w, ptb), (_, univ) in zip(tags_ptb, tags_univ):
        tweet_tag_records.append({
            "Tweet_ID": f"Tweet {idx}",
            "Token": w,
            "Penn_Treebank_Tag": ptb,
            "Universal_Tag": univ
        })

df_twitter_tags = pd.DataFrame(tweet_tag_records)
print("--- Twitter POS Tagging Sample (First 15 Tokens) ---")
print(df_twitter_tags.head(15).to_string(index=False))

--- Twitter POS Tagging Sample (First 15 Tokens) ---
Tweet_ID    Token Penn_Treebank_Tag Universal_Tag
 Tweet 1   loving               VBG          VERB
 Tweet 1      the                DT           DET
 Tweet 1      new                JJ           ADJ
 Tweet 1 #pytorch                NN          NOUN
 Tweet 1      2.0                CD           NUM
 Tweet 1  release                NN          NOUN
 Tweet 1        !                 .             .
 Tweet 1  @openai                JJ           ADJ
 Tweet 1   models               NNS          NOUN
 Tweet 1      run               VBP          VERB
 Tweet 1       so                RB           ADV
 Tweet 1     fast                RB           ADV
 Tweet 1       rn                JJ           ADJ
 Tweet 1        🔥               NNP          NOUN
 Tweet 1        🚀                NN          NOUN


In [4]:
# Distribution of Twitter Universal POS Tags
tag_distribution = df_twitter_tags['Universal_Tag'].value_counts()
print("\n--- Universal POS Tag Distribution across Sample Tweets ---")
print(tag_distribution)


--- Universal POS Tag Distribution across Sample Tweets ---
Universal_Tag
NOUN    13
VERB     7
ADJ      5
.        4
ADV      3
DET      2
NUM      1
PRT      1
ADP      1
Name: count, dtype: int64


# Study and Implementation of POS Tagging, Chunking, and Parsing

## POS Tagging & Chunking using NLTK (RegexpParser) 

In [5]:
# Noun Phrase (NP) and Verb Phrase (VP) Chunking
sentence = "The clever machine learning engineer quickly solved the complex system error."

# 1. Word Tokenization & POS Tagging
sentence_tokens = word_tokenize(sentence)
tagged_sentence = nltk.pos_tag(sentence_tokens)

print("--- POS-Tagged Sentence ---")
print(tagged_sentence)

# 2. Define a Regular Expression Chunk Grammar
# NP: Optional determiner, optional adverbs/adjectives, one or more nouns
# VP: Optional adverb, followed by a verb, followed optionally by a particle or preposition
chunk_grammar = r"""
    NP: {<DT>?<JJ.*|RB>*<NN.*>+}
    VP: {<RB.*>*<VB.*>}
"""

# 3. Create Chunk Parser
chunk_parser = nltk.RegexpParser(chunk_grammar)
chunk_tree = chunk_parser.parse(tagged_sentence)

print("\n--- Generated Chunk Tree Structure ---")
print(chunk_tree)

# 4. Extract Parsed Chunks and IOB Tags
from nltk.chunk import tree2conlltags
iob_tags = tree2conlltags(chunk_tree)

df_chunks = pd.DataFrame(iob_tags, columns=["Token", "POS_Tag", "IOB_Tag"])
print("\n--- IOB Chunk Representations ---")
print(df_chunks.to_string(index=False))

--- POS-Tagged Sentence ---
[('The', 'DT'), ('clever', 'NN'), ('machine', 'NN'), ('learning', 'VBG'), ('engineer', 'JJ'), ('quickly', 'RB'), ('solved', 'VBD'), ('the', 'DT'), ('complex', 'JJ'), ('system', 'NN'), ('error', 'NN'), ('.', '.')]

--- Generated Chunk Tree Structure ---
(S
  (NP The/DT clever/NN machine/NN)
  (VP learning/VBG)
  engineer/JJ
  (VP quickly/RB solved/VBD)
  (NP the/DT complex/JJ system/NN error/NN)
  ./.)

--- IOB Chunk Representations ---
   Token POS_Tag IOB_Tag
     The      DT    B-NP
  clever      NN    I-NP
 machine      NN    I-NP
learning     VBG    B-VP
engineer      JJ       O
 quickly      RB    B-VP
  solved     VBD    I-VP
     the      DT    B-NP
 complex      JJ    I-NP
  system      NN    I-NP
   error      NN    I-NP
       .       .       O


# Chunker with Chinking (Removing Unwanted Chunks)

In [6]:
# Chinking Pattern Example
# Chunk the entire sentence first, then chink (carve out) prepositions and verbs
chink_grammar = r"""
    NP_Chunk:
        {<.*>+}          # Chunk everything
        }<IN|VB.*|CC>+{  # Chink (remove) prepositions, verbs, and conjunctions
"""
chink_parser = nltk.RegexpParser(chink_grammar)
chink_tree = chink_parser.parse(tagged_sentence)

print("--- Tree after Chinking ---")
print(chink_tree)

--- Tree after Chinking ---
(S
  (NP_Chunk The/DT clever/NN machine/NN)
  learning/VBG
  (NP_Chunk engineer/JJ quickly/RB)
  solved/VBD
  (NP_Chunk the/DT complex/JJ system/NN error/NN ./.))


# Dependency Parsing and POS Tagging using spaCy

In [7]:
# spaCy POS Tagging and Dependency Parsing
nlp = spacy.load("en_core_web_sm")

doc = nlp("The clever machine learning engineer quickly solved the complex system error.")

parsed_data = []
for token in doc:
    parsed_data.append({
        "Token": token.text,
        "Lemma": token.lemma_,
        "POS (Fine)": token.tag_,
        "POS (Coarse)": token.pos_,
        "Dependency": token.dep_,
        "Head": token.head.text
    })

df_spacy_parsed = pd.DataFrame(parsed_data)
print("--- spaCy Detailed Syntactic & Dependency Analysis ---")
print(df_spacy_parsed.to_string(index=False))

--- spaCy Detailed Syntactic & Dependency Analysis ---
   Token    Lemma POS (Fine) POS (Coarse) Dependency     Head
     The      the         DT          DET        det  machine
  clever   clever         JJ          ADJ       amod  machine
 machine  machine         NN         NOUN   compound engineer
learning    learn        VBG         VERB        acl  machine
engineer engineer         NN         NOUN      nsubj   solved
 quickly  quickly         RB          ADV     advmod   solved
  solved    solve        VBD         VERB       ROOT   solved
     the      the         DT          DET        det    error
 complex  complex         JJ          ADJ       amod    error
  system   system         NN         NOUN   compound    error
   error    error         NN         NOUN       dobj   solved
       .        .          .        PUNCT      punct   solved


In [8]:
# Extracting Noun Chunks with spaCy
print("\n--- Built-in spaCy Noun Chunks ---")
for chunk in doc.noun_chunks:
    print(f"Phrase: {chunk.text:<35} | Root: {chunk.root.text:<10} | Dependency: {chunk.root.dep_}")

# Visualize dependency graph (in a Jupyter environment)
displacy.render(doc, style="dep", jupyter=True, options={"distance": 110})


--- Built-in spaCy Noun Chunks ---
Phrase: The clever machine learning engineer | Root: engineer   | Dependency: nsubj
Phrase: the complex system error            | Root: error      | Dependency: dobj


### Method Comparison: NLTK vs. spaCy (Tagging, Chunking, and Parsing)

| Metric / Aspect | NLTK (RegexpParser & Perceptron Tagger) | spaCy (`en_core_web_sm` Pipeline) |
| :--- | :--- | :--- |
| **Primary Architecture** | Rule-based regular expressions over statistical POS tags | Transition-based deep convolutional / transformer neural network |
| **Tagging Granularity** | Supports Penn Treebank (fine-grained) and Universal POS (coarse-grained) | Simultaneous access to fine tags (`token.tag_`) and Universal tags (`token.pos_`) |
| **Parsing Type** | Shallow parsing (Chunking via Regex patterns; flat IOB format) | Full dependency parsing (directed head-dependent trees) + built-in noun chunking |
| **Grammar Customization** | Highly flexible user-defined grammars using regex and chinking | Rule-based entity/pattern matching via `Matcher` and custom pipeline extensions |
| **Twitter / Social Media Handling** | Dedicated `TweetTokenizer` preserves handles, emojis, and hashtags | Standard statistical tokenizer (requires fine-tuning/custom components for slang) |
| **Output Representation** | Hierarchical text tree (`nltk.Tree`) or IOB tagged tuples | Direct access via token attributes (`dep_`, `head`) and interactive SVG (`displacy`) |
| **Execution Speed & Scalability** | Fast on small corpora; regex parsing scales linearly with rule complexity | Optimized C-level (Cython) execution designed for high-throughput production pipelines |
| **Best Use Case** | Teaching syntactic rules, lightweight chunk extraction, and custom grammar development | Industrial NLP, complete syntactic parsing, and large-scale text analysis |

### Lab Exercises: POS Tagging, Chunking, and Parsing

1. **Custom Prepositional Phrase (PP) Chunk Grammar:**
   * Write an NLTK regular expression chunk grammar to identify Prepositional Phrases ($\text{PP}$), defined as a preposition followed by a Noun Phrase ($\text{NP}$):
     $$\text{PP: } \{<\text{IN}><\text{DT}>?<\text{JJ}>*<\text{NN.*}>+\}$$
   * Test your grammar on the sentence:  
     `"The scientist conducted tests in the laboratory during the night."`
   * Print the parsed chunk tree and extract all detected prepositional phrases.

2. **Social Media vs. Standard Text POS Tagging:**
   * Take the following tweet containing slang, hashtags, mentions, and links:  
     `"omg @user this is sooo cool!! 🔥 https://example.com #nlp"`
   * Tokenize and tag the sentence twice using:
     1. Standard NLTK `word_tokenize` + `pos_tag`
     2. NLTK `TweetTokenizer` + `pos_tag`
   * Display the outputs side-by-side in a table and document how URLs, mentions, and expressive punctuation are partitioned differently.

3. **Dependency Parsing Triplet Extraction (Subject-Verb-Object):**
   * Using `spaCy`, write a reusable function `extract_svo_triplets(doc)` that analyzes dependency parse trees.
   * Identify and extract `(Subject, Verb, Object)` tuples based on:
     * Nominal subject relations (`nsubj`)
     * Root action verbs (`VERB`)
     * Direct object relations (`dobj`)
   * Test your function on at least three distinct sentences and print the resulting triplets.

4. **Chinking Implementation (Unwanted Pattern Exclusion):**
   * Design a chunk parser with a **chinking pattern** that initially captures all continuous words and carves out ("chinks") determiners (`DT`), personal pronouns (`PRP`), and coordinating conjunctions (`CC`).
   * Apply it to the sentence:  
     `"The clever machine learning engineer quickly solved the complex system error."`
   * Convert the resulting parse tree into **IOB tags** using NLTK's `tree2conlltags` and display the final token-tag-chunk dataframe.